# Between Words

*Words Hold Worlds* gave seven words a world each. This notebook looks for the
worlds **between** them.

A text-to-image model never reads your prompt as words. It turns the prompt into
an **embedding**, a grid of numbers (77 tokens × 1024 numbers each), and draws
from that. Replicate only lets you send text, so you only ever get the seven
worlds. Here we hold the numbers ourselves, so we can blend two prompts' embeddings
and draw what sits halfway between *glow* and *bread*, a scene no prompt could
name.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the
cells top to bottom.

- **Sections 1–2** load the model and show some in-between images right here.
- **Section 3** renders the whole chain of words as frames for the site.
- **Section 4** serves the model through ngrok, so the site can make new
  in-betweens live. That makes this notebook your own small Replicate.

## 1. Setup

In [ ]:
!pip -q install diffusers transformers accelerate fastapi uvicorn pyngrok

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

# sd-turbo draws a 512px image in a single denoising step, fast enough to scrub
# live on Colab's free T4. For a slower, richer model, use
# "stable-diffusion-v1-5/stable-diffusion-v1-5" with STEPS = 25, GUIDANCE = 7.5.
MODEL_ID = "stabilityai/sd-turbo"
STEPS = 1
GUIDANCE = 0.0
SIZE = 512
SEED = 7  # every frame starts from the same noise, so only the words change

assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU, then run again."

pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, variant="fp16"
).to("cuda")
pipe.set_progress_bar_config(disable=True)

## 2. The words, and the space between them

These are the seven world words from *Words Hold Worlds*, in their order from
cold to hope. Each prompt is that word's scene brief from
`words-hold-worlds/src/worlds.json`, plus a shared style so every frame looks
like it belongs to the same piece. A word that isn't in the list gets a plain
prompt of its own, so you can try any word.

In [ ]:
STYLE = "Monochrome pointillism, white points of light on black, soft fog, film grain. "

# Copied from words-hold-worlds/src/worlds.json. Keep the two in step.
SCENES = {
    "rain": "Cold. Dense sheets of white dots falling all around you, a small figure in an oversized hunting jacket walking through them, shoulders hunched.",
    "puddle": "Defeat. The camera at the waterline of a mud puddle. A bundle of baby clothes half sunk in it, rings of dots spreading out from where it fell, and her wavering reflection upside down in the water.",
    "glow": "Longing. From the dark lane, an open kitchen door: the bakery's ovens inside, and a fan of dense bright dots spilling out across the mud, with rain falling through the light.",
    "bread": "Kindness. A girl slumped at the base of an apple tree, a pig pen and pig beside her, two loaves arcing through the air toward her. The burned crusts are the densest dots in the scene.",
    "pig": "Cruelty and courage. Close at the trough: chunks of burned bread tumbling into it and the pig eating, while behind the fence a boy, back turned, tears at the loaves.",
    "fluffy": "Relief. Lying on your back on the school yard grass, looking up: soft, slow clouds of dots drifting overhead in warm air.",
    "dandelion": "Hope. A single dandelion head close up, its seeds lifting off one by one and drifting past and around you, up into the dark.",
}
CHAIN = list(SCENES)  # the order the site walks through


def clean(word):
    return word.strip().lower()[:60]


def prompt_for(word):
    return STYLE + SCENES.get(word, f"The world held inside the word '{word}'.")

**Embedding** is the step Replicate hides. `embed` runs only the model's text
encoder and hands back the numbers.

**Blending.** `slerp` (spherical interpolation) moves from one embedding to
the other along the surface of a sphere, not a straight line. In this many
dimensions, a straight-line midpoint comes out shorter than either end. The
model has rarely seen embeddings that short, and the middle frames turn to mush.
Going around the sphere keeps every in-between the size of a real prompt.

**The same noise.** Every image starts from random noise. Fixing it (`NOISE`)
means the only thing changing between frames is the embedding, so the frames
morph instead of jumping to new compositions.

In [ ]:
_embeddings = {}


@torch.no_grad()
def embed(word):
    if word not in _embeddings:
        _embeddings[word], _ = pipe.encode_prompt(prompt_for(word), "cuda", 1, False)
    return _embeddings[word]


def slerp(a, b, t, eps=1e-7):
    a_flat, b_flat = a.flatten().float(), b.flatten().float()
    dot = (a_flat / a_flat.norm()) @ (b_flat / b_flat.norm())
    omega = torch.acos(dot.clamp(-1, 1))  # the angle between the two embeddings
    if omega < eps:
        mixed = torch.lerp(a_flat, b_flat, t)
    else:
        mixed = (torch.sin((1 - t) * omega) * a_flat + torch.sin(t * omega) * b_flat) / torch.sin(omega)
    return mixed.reshape(a.shape).to(a.dtype)


latent = SIZE // pipe.vae_scale_factor
NOISE = torch.randn(
    (1, pipe.unet.config.in_channels, latent, latent),
    generator=torch.Generator("cuda").manual_seed(SEED),
    device="cuda",
    dtype=torch.float16,
)


@torch.no_grad()
def between(a, b, t):
    # t = 0 is word a, t = 1 is word b, anything else is a world in between.
    mixed = slerp(embed(clean(a)), embed(clean(b)), t)
    return pipe(
        prompt_embeds=mixed,
        latents=NOISE.clone(),
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        height=SIZE,
        width=SIZE,
    ).images[0]

### Try it

Six steps from *bread* to *rain*. Change the words and run it again. Words
outside the seven work too: `strip("bread", "ocean")`.

In [ ]:
from PIL import Image
from IPython.display import display


def strip(a, b, n=6):
    thumb = SIZE // 3
    sheet = Image.new("RGB", (thumb * n, thumb))
    for i in range(n):
        sheet.paste(between(a, b, i / (n - 1)).resize((thumb, thumb)), (i * thumb, 0))
    return sheet


display(strip("bread", "rain"))

## 3. Render the chain for the site

This walks the whole chain (rain → puddle → glow → bread → pig → fluffy →
dandelion) and saves every frame, along with a `manifest.json` the page reads.
It takes a few minutes on a T4.

When the zip downloads, unzip its contents into `public/between-words/frames/`
in the repo, so you end up with `public/between-words/frames/manifest.json`
next to `000.webp`, `001.webp`, and so on.

In [ ]:
import json, os, shutil

FRAMES_PER_PAIR = 24
OUT = "frames"

shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT)

n = 0
for i in range(len(CHAIN) - 1):
    a, b = CHAIN[i], CHAIN[i + 1]
    # Each pair's last frame is the next pair's first, so only the first pair keeps frame 0.
    for k in range(0 if i == 0 else 1, FRAMES_PER_PAIR):
        between(a, b, k / (FRAMES_PER_PAIR - 1)).save(f"{OUT}/{n:03d}.webp", quality=80)
        n += 1
    print(f"{a} → {b}")

with open(f"{OUT}/manifest.json", "w") as f:
    json.dump(
        {
            "words": CHAIN,
            "framesPerPair": FRAMES_PER_PAIR,
            "count": n,
            "size": SIZE,
            "model": MODEL_ID,
            "seed": SEED,
            "style": STYLE,
        },
        f,
        indent=2,
    )

shutil.make_archive("between-words-frames", "zip", OUT)

from google.colab import files
files.download("between-words-frames.zip")

## 4. Serve it live: your own Replicate

This turns the notebook into a small web server, and ngrok gives it a public
address so the site can reach it. On the page, you can then type **any two
words** and scrub between them, with each frame drawn on this GPU.

**One-time setup:**
1. Sign up at [ngrok.com](https://ngrok.com) and copy your authtoken from the
   dashboard.
2. In Colab's left sidebar, open the **key icon (Secrets)**, add a secret named
   `NGROK_AUTHTOKEN` with the token as its value, and turn on *Notebook access*.

Run both cells and open the link the second one prints. The server lives only as long as
this runtime. When Colab disconnects, the site goes back to the frames from
section 3.

In [ ]:
# Colab forgets everything when its runtime restarts: installed packages and
# the loaded model alike. Reinstall what this section needs (quick if it's
# already there), and stop early if the model from sections 1–2 is gone.
!pip -q install fastapi uvicorn pyngrok

if "between" not in globals():
    raise RuntimeError(
        "The model isn't loaded. The runtime has probably restarted since you "
        "last ran this notebook. Run sections 1 and 2 again, then this cell. "
        "(Section 3 can be skipped.)"
    )

In [ ]:
import io, threading, time

import uvicorn
from fastapi import FastAPI, Query, Response
from fastapi.middleware.cors import CORSMiddleware
from google.colab import userdata
from pyngrok import ngrok

SITE = "https://jackienamdesign.github.io/Shared-Minds/between-words/"
PORT = 8000

app = FastAPI()
# The page lives on github.io and this server on ngrok, so the browser needs
# this server's permission to read its replies.
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["GET"], allow_headers=["*"])
gpu = threading.Lock()  # one image at a time on the GPU


@app.get("/health")
def health():
    return {"model": MODEL_ID, "words": CHAIN, "size": SIZE}


@app.get("/frame")
def frame(a: str, b: str, t: float = Query(0.5, ge=0, le=1)):
    with gpu:
        image = between(a, b, round(t, 3))
    buffer = io.BytesIO()
    image.save(buffer, "WEBP", quality=80)
    return Response(buffer.getvalue(), media_type="image/webp")


# Running this cell again replaces the old server and tunnel instead of colliding with them.
if "server" in globals():
    server.should_exit = True
    time.sleep(1)
ngrok.kill()

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
public_url = ngrok.connect(PORT).public_url

print("Server:", public_url)
print("Open the piece live:", f"{SITE}?live={public_url}")
print("Or locally:", f"http://localhost:5173/Shared-Minds/between-words/?live={public_url}")

When you're done: **Runtime → Disconnect and delete runtime.** That shuts down
the server and frees the GPU.